In [ ]:
# Titanic preprocessing: handle all nulls, encode categoricals, scale numerics. Print before/after
# null counts. Show distributions.

import pandas as pd

In [ ]:
df = pd.read_csv("../data/raw/titanic.csv")

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

In [ ]:
# encode this titanic data

df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df = pd.get_dummies(df, columns=['Sex', 'Embarked'], drop_first=True)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load Titanic dataset
df = pd.read_csv("../data/raw/titanic.csv")

# --- Before preprocessing: null counts ---
print("Null counts before preprocessing:\n", df.isnull().sum())

# Handle missing values
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
df['Cabin'].fillna("Unknown", inplace=True)

# Encode categorical variables
categorical_cols = ['Sex', 'Embarked', 'Cabin']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Scale numerical variables
numeric_cols = ['Age', 'Fare']
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# --- After preprocessing: null counts ---
print("\nNull counts after preprocessing:\n", df.isnull().sum())

# Show distributions before/after scaling
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.histplot(pd.read_csv("../data/raw/titanic.csv")['Age'], bins=30, ax=axes[0,0], kde=True)
axes[0,0].set_title("Age (Before)")

sns.histplot(df['Age'], bins=30, ax=axes[0,1], kde=True)
axes[0,1].set_title("Age (After Scaling)")

sns.histplot(pd.read_csv("../data/raw/titanic.csv")['Fare'], bins=30, ax=axes[1,0], kde=True)
axes[1,0].set_title("Fare (Before)")

sns.histplot(df['Fare'], bins=30, ax=axes[1,1], kde=True)
axes[1,1].set_title("Fare (After Scaling)")

plt.tight_layout()
plt.show()


In [ ]:
# 2.1.3 Encoding Categorical Variables
# Python
# Label Encoding — for ordinal data (has order: Small < Medium < Large)
size_map={'Small':0,'Medium':1,'Large':2,'XL':3}
df['Size_enc']=df['Size'].map(size_map)
# One-Hot Encoding — for nominal data (no order)
df=pd.get_dummies(df, columns=['Sex','Embarked'], drop_first=True)
# Sklearn OHE (for pipelines)
from sklearn.preprocessing import OneHotEncoder
ohe=OneHotEncoder(sparse_output=False,drop='first',handle_unknown='ignore')
encoded=ohe.fit_transform(df[['Sex']])
# Target Encoding — mean of target per category (use carefully!)
target_means=df.groupby('Pclass')['Survived'].mean()

In [ ]:
# Python
# ---- Titanic Feature Engineering ---
df['FamilySize']=df['SibSp']+df['Parch']+1
df['IsAlone']=(df['FamilySize']==1).astype(int)
# Extract title from name
df['Title']=df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
rare=['Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir']
df['Title']=df['Title'].replace(rare,'Rare')
df['Title']=df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
# Age bins
df['AgeBin']=pd.cut(df['Age'],bins=[0,12,18,35,60,100],
labels=['Child','Teen','Adult','Middle','Senior'])
df['FarePerPerson']=df['Fare']/df['FamilySize'].replace(0,1)
df['Deck']=df['Cabin'].str[0].fillna('U')
df['Age_x_Fare']=df['Age']*df['Fare']
# 2.2.1 Outlier Detection & Treatme

In [ ]:
# Family size
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Title from name
df["Title"] = df["Name"].str.extract(" ([A-Za-z]+)\.", expand=False)

# Fare per person
df["Fare_per_person"] = df["Fare"] / df["FamilySize"]

# Age bins
df["Age_bin"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 40, 60, 80],
    labels=["Child", "Teen", "Adult", "MiddleAge", "Senior"],
)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

num_features = ['Age','Fare']
cat_features = ['Sex','Embarked','Pclass']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(), cat_features)
])

pipeline = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression())
])

X_train = Xtr[num_features + cat_features].copy()
y_train = ytr.copy()
X_train['Age'] = X_train['Age'].fillna(X_train['Age'].median())
X_train['Embarked'] = X_train['Embarked'].fillna(X_train['Embarked'].mode()[0])
pipeline.fit(X_train, y_train)
joblib.dump(pipeline, "titanic_pipeline.pkl")

# Load + predict
model = joblib.load("titanic_pipeline.pkl")
manual_passengers = pd.DataFrame([...])  # 5 records
print(model.predict(manual_passengers))
